# Importing Necessary Libraries:

In [2]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Conv1D, Dense, LSTM, GRU, TextVectorization, Embedding
from tensorflow.keras.layers import GlobalAveragePooling1D, GlobalMaxPool1D, Bidirectional
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

import tensorflow_hub as hub
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Downloading Helper Functions Script

In [3]:
# Run the below lines if helper functions are not already downloaded:
# Downloading Helper Functions:

#import wget
#url = 'https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py'
#file = wget.download(url)

In [4]:
from helper_functions import calculate_results

## Imprting The Data:

In [5]:
# Getting Data From Github:
!git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git

fatal: destination path 'pubmed-rct' already exists and is not an empty directory.


In [6]:
# Data Directory:
data_dir = 'pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign/'

In [7]:
# Checking Contents of Data Directory:
os.listdir(data_dir)

['test.txt', 'dev.txt', 'train.txt']

## Creating Functions to Pre-Process The Data

In [10]:
# Function to Read Lines from Files in Data Directory:
def get_lines(filename):
    """
      Reads filename (a text file) and returns the lines of text as a list.

      Args:
          filename: a string containing the target filepath to read.

      Returns:
          A list of strings with one string per line from the target filename.
          For example:
          ["this is the first line of filename",
           "this is the second line of filename",
           "..."]
    """

    with open(filename, mode= 'r') as f:
        return f.readlines()

In [11]:
train_lines = get_lines(data_dir + 'train.txt')
train_lines[:20]

['###24293578\n',
 'OBJECTIVE\tTo investigate the efficacy of @ weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at @ weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .\n',
 'METHODS\tA total of @ patients with primary knee OA were randomized @:@ ; @ received @ mg/day of prednisolone and @ received placebo for @ weeks .\n',
 'METHODS\tOutcome measures included pain reduction and improvement in function scores and systemic inflammation markers .\n',
 'METHODS\tPain was assessed using the visual analog pain scale ( @-@ mm ) .\n',
 'METHODS\tSecondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis Index scores , patient global assessment ( PGA ) of the severity of knee OA , and @-min walk distance ( @MWD ) .\n',
 'METHODS\tSerum levels of interleukin @ ( IL-@ ) , IL-@ , tumor necrosis factor ( TNF ) - , and 

In [21]:
# Function to Pre-Process The Data in Train, Test and Validation Files:

def preprocess_text_with_line_numbers(filename):
    """Returns a list of dictionaries of abstract line data.

      Takes in filename, reads its contents and sorts through each line,
      extracting things like the target label, the text of the sentence,
      how many sentences are in the current abstract and what sentence number
      the target line is.

      Args:
          filename: a string of the target text file to read and extract line data
          from.

      Returns:
          A list of dictionaries each containing a line from an abstract,
          the lines label, the lines position in the abstract and the total number
          of lines in the abstract where the line is from. For example:

          [{"target": 'CONCLUSION',
            "text": The study couldn't have gone better, turns out people are kinder than you think",
            "line_number": 8,
            "total_lines": 8}]
    """

    input_data = get_lines(filename)
    abstract_lines = ""
    abstract_samples = []

    for line in input_data:

        if line.startswith("###"):
          abstract_id = line
          abstract_lines = ""

        elif line.isspace():
            abstract_line_split = abstract_lines.splitlines()

            for abstract_line_number, abstract_line in enumerate(abstract_line_split):
                line_data = {}
                target_text_split = abstract_line.split("\t")
                line_data["Target"] = target_text_split[0]
                line_data["Text"] = target_text_split[1]
                line_data["Line_Number"] = abstract_line_number
                line_data["Total_Lines"] = len(abstract_line_split) - 1
                abstract_samples.append(line_data)

        else:
            abstract_lines += line

    return abstract_samples

In [14]:
# Getting All File Names from Data Directory in a list:
filenames = [data_dir + filename for filename in os.listdir(data_dir)]

In [15]:
filenames

['pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign/test.txt',
 'pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign/dev.txt',
 'pubmed-rct/PubMed_20k_RCT_numbers_replaced_with_at_sign/train.txt']

In [22]:
# Getting Data from All files:
train_data = preprocess_text_with_line_numbers(filename= filenames[2])
val_data = preprocess_text_with_line_numbers(filename= filenames[1])
test_data = preprocess_text_with_line_numbers(filename= filenames[0])

In [23]:
train_data[:5]

[{'Target': 'OBJECTIVE',
  'Text': 'To investigate the efficacy of @ weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at @ weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .',
  'Line_Number': 0,
  'Total_Lines': 11},
 {'Target': 'METHODS',
  'Text': 'A total of @ patients with primary knee OA were randomized @:@ ; @ received @ mg/day of prednisolone and @ received placebo for @ weeks .',
  'Line_Number': 1,
  'Total_Lines': 11},
 {'Target': 'METHODS',
  'Text': 'Outcome measures included pain reduction and improvement in function scores and systemic inflammation markers .',
  'Line_Number': 2,
  'Total_Lines': 11},
 {'Target': 'METHODS',
  'Text': 'Pain was assessed using the visual analog pain scale ( @-@ mm ) .',
  'Line_Number': 3,
  'Total_Lines': 11},
 {'Target': 'METHODS',
  'Text': 'Secondary outcome measures included the Western Ontari

In [24]:
val_data[:5]

[{'Target': 'BACKGROUND',
  'Text': 'IgE sensitization to Aspergillus fumigatus and a positive sputum fungal culture result are common in patients with refractory asthma .',
  'Line_Number': 0,
  'Total_Lines': 9},
 {'Target': 'BACKGROUND',
  'Text': 'It is not clear whether these patients would benefit from antifungal treatment .',
  'Line_Number': 1,
  'Total_Lines': 9},
 {'Target': 'OBJECTIVE',
  'Text': 'We sought to determine whether a @-month course of voriconazole improved asthma-related outcomes in patients with asthma who are IgE sensitized to A fumigatus .',
  'Line_Number': 2,
  'Total_Lines': 9},
 {'Target': 'METHODS',
  'Text': 'Asthmatic patients who were IgE sensitized to A fumigatus with a history of at least @ severe exacerbations in the previous @ months were treated for @ months with @ mg of voriconazole twice daily , followed by observation for @ months , in a double-blind , placebo-controlled , randomized design .',
  'Line_Number': 3,
  'Total_Lines': 9},
 {'Targe

In [25]:
test_data[:5]

[{'Target': 'BACKGROUND',
  'Text': 'This study analyzed liver function abnormalities in heart failure patients admitted with severe acute decompensated heart failure ( ADHF ) .',
  'Line_Number': 0,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'A post hoc analysis was conducted with the use of data from the Evaluation Study of Congestive Heart Failure and Pulmonary Artery Catheterization Effectiveness ( ESCAPE ) .',
  'Line_Number': 1,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'Liver function tests ( LFTs ) were measured at @ time points from baseline , at discharge , and up to @ months follow-up .',
  'Line_Number': 2,
  'Total_Lines': 8},
 {'Target': 'RESULTS',
  'Text': 'Survival analyses were used to assess the association between admission Model of End-Stage Liver Disease Excluding International Normalized Ratio ( MELD-XI ) scores and patient outcome.There was a high prevalence of abnormal baseline ( admission ) LFTs ( albumin @ % , aspartate transaminase @ %

In [26]:
# Creeating Dataframes for Training, Validation and Test Data:
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
test_df = pd.DataFrame(test_data)

In [27]:
train_df.head(10)

,Target,Text,Line_Number,Total_Lines
0,OBJECTIVE,To investigate the efficacy of @ weeks of dail...,0,11
1,METHODS,A total of @ patients with primary knee OA wer...,1,11
2,METHODS,Outcome measures included pain reduction and i...,2,11
3,METHODS,Pain was assessed using the visual analog pain...,3,11
4,METHODS,Secondary outcome measures included the Wester...,4,11
5,METHODS,"Serum levels of interleukin @ ( IL-@ ) , IL-@ ...",5,11
6,RESULTS,There was a clinically relevant reduction in t...,6,11
7,RESULTS,The mean difference between treatment arms ( @...,7,11
8,RESULTS,"Further , there was a clinically relevant redu...",8,11
9,RESULTS,These differences remained significant at @ we...,9,11
